In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def load_model_and_tokenizer(
    model_name="meta-llama/Llama-2-7b-chat-hf",
    device="cuda",
    load_4bit=False,
):
    """
    加载 tokenizer 和模型（可选 4bit 量化），返回 model 和 tokenizer。
    """
    quant_cfg = BitsAndBytesConfig(load_in_4bit=True) if load_4bit else None

    tokenizer = AutoTokenizer.from_pretrained(
        model_name, use_fast=True, trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto" if device == "auto" else None,
        quantization_config=quant_cfg,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    ).eval()

    model.to(device)
    return model, tokenizer


def apply_mask(model: torch.nn.Module, mask_path: str):
    """
    应用 mask（将 mask == True 的位置置零），并返回每层和整体 sparsity 信息。
    """
    if not os.path.isfile(mask_path):
        raise FileNotFoundError(f"Mask file not found: {mask_path}")

    print(f"[INFO] Loading pruning mask from: {mask_path}")
    mask_dict = torch.load(mask_path, map_location="cpu")

    sparsity_stats = {}
    total_pruned = total_params = 0

    with torch.no_grad():
        for name, param in model.named_parameters():
            if not name.endswith(".weight"):
                continue  # 通常只对 weight 做剪枝

            module_name = name.rsplit(".weight", 1)[0]  # 匹配 mask_dict 的 key
            if module_name in mask_dict:
                mask = mask_dict[module_name].to(param.device, dtype=torch.bool)
                if mask.shape != param.shape:
                    raise ValueError(f"Mask shape {mask.shape} does not match param {name} shape {param.shape}")

                pruned = mask.sum().item()
                total = mask.numel()
                sparsity = 100.0 * pruned / (total + 1e-6)

                param.data[mask] = 0.0  # 应用 mask

                sparsity_stats[name] = {
                    "pruned": pruned,
                    "total": total,
                    "sparsity": sparsity,
                }

                total_pruned += pruned
                total_params += total

    print(f"\n[SUMMARY] Sparsity per parameter:")
    for name, stat in sparsity_stats.items():
        print(f"  {name:<60} -> {stat['sparsity']:.2f}% ({stat['pruned']:,}/{stat['total']:,})")

    print(f"\n[GLOBAL] Total Sparsity: {100.0 * total_pruned / total_params:.2f}% "
          f"({total_pruned:,}/{total_params:,})")
    return sparsity_stats


# 示例使用：
if __name__ == "__main__":
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    mask_path = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/unstructured/wanda_weightonly/GSM8K_direct_120/FT_mask/mask_bottom_0.100.pt"
    device = "cuda:5"

    model, tokenizer = load_model_and_tokenizer(model_name, device)
    sparsity_stats = apply_mask(model, mask_path)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Loading pruning mask from: /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/unstructured/wanda_weightonly/GSM8K_direct_120/FT_mask/mask_bottom_0.100.pt

[SUMMARY] Sparsity per parameter:
  model.embed_tokens.weight                                    -> 100.00% (131,069,927/131,072,000)
  model.layers.0.self_attn.q_proj.weight                       -> 90.01% (15,101,952/16,777,216)
  model.layers.0.self_attn.k_proj.weight                       -> 90.01% (15,101,952/16,777,216)
  model.layers.0.self_attn.v_proj.weight                       -> 90.01% (15,101,952/16,777,216)
  model.layers.0.self_attn.o_proj.weight                       -> 90.01% (15,101,952/16,777,216)
  model.layers.0.mlp.gate_proj.weight                          -> 90.01% (40,586,496/45,088,768)
  model.layers.0.mlp.up_proj.weight                            -> 90.01% (40,586,496/45,088,768)
  model.layers.0.mlp.down_proj.weight                          -> 90.01% (40,583,168

<generator object Module.named_parameters at 0x7fe205440970>

In [ ]:
sparsity_stats = apply_mask(model, mask_path)

>> The sky appears blue because of a phenomenon called Rayleigh scattering, which is the scattering of light or other electromagnetic radiation by small particles, such as molecules of gases. When sunlight enters Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen and oxygen. These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is known as Rayleigh scattering.

The reason for this preference for blue light has to do with the way that the molecules of the gas interact with the light. The blue light has a shorter wavelength, which means that it has more energy. This energy is what is scattered by the gas molecules, causing them to redirect the light in different directions. The longer wavelengths of red light, on the other hand, have less energy and are scattered less by the gas molecules, allowing them to travel in a more direct path to the observer.

As a result of this scatterin